In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import urllib.parse

from sklearn.metrics.pairwise import cosine_similarity

### Loading Animelist

In [2]:
df_AnimeList = pd.DataFrame()
for f in list(Path('../Scraping/Season').rglob('*.csv')):
    if re.search(r'.ipynb_checkpoints', str(f)) is not None:
        continue

    df_Read = pd.read_csv(f)
    df_AnimeList = pd.concat([df_AnimeList, df_Read])

del df_Read

In [3]:
df_AnimeList['MAL_id'] = df_AnimeList['Link'].apply(lambda x : int(re.findall(r'anime/(\d+)',x)[0]))
df_AnimeList = df_AnimeList.drop(columns=['Link', 'Unnamed: 0'])
df_AnimeList = df_AnimeList.set_index('MAL_id')
df_AnimeList = df_AnimeList.drop_duplicates()
df_AnimeList

,Title
MAL_id,
30,Shinseiki Evangelion
852,Gokinjo Monogatari
495,Bakuretsu Hunters
1567,Kaitou Saint Tail
116,Shinpi no Sekai El-Hazard (TV)
...,...
54145,Cardfight!! Vanguard: D Series Season 9
53722,Bullet/Bullet
60726,My Melody & Kuromi


### Loading User Ratings

In [4]:
df_Raw = pd.DataFrame()
for f in list(Path('../Scraping/Users/Watchlist').rglob('*.csv')):
    if re.search(r'.ipynb_checkpoints', str(f)) is not None:
        continue

    df_Read = pd.read_csv(f)
    df_Read['User'] = f.name.replace('.csv', '')
    df_Raw = pd.concat([df_Raw, df_Read])

del df_Read

In [5]:
df_Raw = df_Raw.drop(columns=['Unnamed: 0'])
df_Raw = df_Raw.dropna()

df_Raw['MAL_id'] = df_Raw['MAL_id'].astype(int)
df_Raw['Title'] = df_Raw['Title'].apply(lambda x : urllib.parse.unquote(x).replace('_', ' '))
df_Raw

,MAL_id,Title,Score,Type,User
3,44511,Chainsaw Man,9.0,TV,-Ellisson-
8,50265,Spy x Family,9.0,TV,-Ellisson-
9,1689,Byousoku 5 Centimeter,7.0,MOVIE,-Ellisson-
10,35839,Sora yori mo Tooi Basho,8.0,TV,-Ellisson-
11,28851,Koe no Katachi,9.0,MOVIE,-Ellisson-
...,...,...,...,...,...
265,42963,Kanojo Okarishimasu 2nd Season,3.0,TV,_Theodore
266,42205,Shaman King 2021,5.0,TV,_Theodore
267,46093,Shiroi Suna no Aquatope,5.0,TV,_Theodore
268,46471,Tantei wa Mou Shindeiru,3.0,TV,_Theodore


### User-Anime Score Matrix

In [6]:
matrix_Raw = df_Raw.pivot_table(index='User', columns='MAL_id', values='Score', aggfunc='mean')
matrix_Raw = matrix_Raw.fillna(0)
matrix_Raw = matrix_Raw.astype(np.int8)
matrix_Raw

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60512,60544,60553,60627,60629,60736,60749,60772,61130,61134
User,,,,,,,,,,,,,,,,,,,,,
-Ellisson-,8,0,0,0,0,0,0,0,0,10,...,0,0,0,0,0,0,0,0,0,0
-HonokaKousaka-,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
-Norza-,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
-Trinody-,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
-Xythus-,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zenjamibu,0,0,10,0,0,0,0,0,10,0,...,0,0,0,0,0,0,0,0,0,0
zhorahh,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
zikangue,7,0,0,0,0,9,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### User-User Similarity Matrix

In [7]:
matrix_Users = pd.DataFrame(cosine_similarity(matrix_Raw), index=matrix_Raw.index, columns=matrix_Raw.index)
matrix_Users.head(10)

User,-Ellisson-,-HonokaKousaka-,-Norza-,-Trinody-,-Xythus-,0_Kira_0,0neiros,0rekiHoutarou,123TeejeeyBey,1fallenleaf,...,yumenade,yutsuru,zairiko,zarizz,zazanya,zenjamibu,zhorahh,zikangue,zilch16,zwrei
User,,,,,,,,,,,,,,,,,,,,,
-Ellisson-,1.000000,0.318937,0.262912,0.222723,0.371772,0.264173,0.263367,0.232079,0.000000,0.285285,...,0.049775,0.081895,0.165761,0.126819,0.235197,0.055872,0.125442,0.330626,0.069273,0.103906
-HonokaKousaka-,0.318937,1.000000,0.198153,0.238584,0.349097,0.271575,0.262889,0.322875,0.033490,0.180589,...,0.042189,0.074047,0.175212,0.106268,0.150859,0.111218,0.080724,0.200075,0.060039,0.052585
-Norza-,0.262912,0.198153,1.000000,0.305684,0.248460,0.173178,0.162018,0.221289,0.000000,0.092652,...,0.000000,0.018723,0.100901,0.025127,0.156391,0.097872,0.000000,0.254551,0.036748,0.082479
-Trinody-,0.222723,0.238584,0.305684,1.000000,0.379080,0.331789,0.252743,0.360709,0.000000,0.213139,...,0.044986,0.096446,0.160294,0.071228,0.222878,0.080210,0.049068,0.274617,0.089413,0.000000
-Xythus-,0.371772,0.349097,0.248460,0.379080,1.000000,0.437163,0.313810,0.476413,0.116747,0.327651,...,0.038106,0.091664,0.261281,0.085573,0.240923,0.070281,0.137909,0.354235,0.106830,0.041473
0_Kira_0,0.264173,0.271575,0.173178,0.331789,0.437163,1.000000,0.275815,0.488422,0.050571,0.153598,...,0.126029,0.108029,0.295504,0.060080,0.118000,0.067108,0.086132,0.204728,0.152791,0.027679
0neiros,0.263367,0.262889,0.162018,0.252743,0.313810,0.275815,1.000000,0.338728,0.046132,0.156473,...,0.021297,0.091869,0.156020,0.071377,0.096997,0.109626,0.097975,0.201366,0.090689,0.065143
0rekiHoutarou,0.232079,0.322875,0.221289,0.360709,0.476413,0.488422,0.338728,1.000000,0.120647,0.168382,...,0.074367,0.085589,0.247761,0.081374,0.145101,0.126125,0.121759,0.302219,0.134844,0.000000
123TeejeeyBey,0.000000,0.033490,0.000000,0.000000,0.116747,0.050571,0.046132,0.120647,1.000000,0.206826,...,0.000000,0.062493,0.000000,0.000000,0.040321,0.000000,0.163885,0.048932,0.061036,0.000000


## Test User Actual_Hana
Gather Top 10 Most Similar Users

In [8]:
USER = 'Actual_Hana'

similar_users = matrix_Users.loc[USER].sort_values(ascending=False)[1:11]
similar_users

User
_Mei_            0.407530
jasmine_kb08     0.392706
xiuzuki          0.385509
MysticMira       0.374602
gellai           0.365401
Chaos_Griever    0.360107
kannequi         0.339587
xkourokox        0.337709
Aohau            0.336381
Jer1ch0          0.333218
Name: Actual_Hana, dtype: float64

## Shows User Watched. Others have Watched Some

In [9]:
alreadyWatched = matrix_Raw.loc[USER] > 0
matrix_Raw.loc[np.append(USER, similar_users.index), alreadyWatched]

MAL_id,20,22,121,123,226,270,355,356,384,853,...,22145,22199,22319,22535,23273,23283,23317,23673,27899,28223
User,,,,,,,,,,,,,,,,,,,,,
Actual_Hana,10,9,10,10,9,9,9,9,10,9,...,8,9,10,10,10,10,8,10,10,10
_Mei_,0,0,9,0,0,9,0,7,0,9,...,0,7,9,8,9,8,0,0,6,0
jasmine_kb08,0,0,0,7,7,0,7,6,0,0,...,0,0,0,0,0,0,0,0,0,0
xiuzuki,10,0,6,0,0,0,10,0,0,8,...,0,8,8,0,8,0,0,0,9,0
MysticMira,4,0,0,0,0,0,0,0,0,6,...,8,7,7,8,8,0,0,0,9,0
gellai,0,0,0,0,4,0,0,0,0,10,...,0,0,7,0,7,0,0,0,0,0
Chaos_Griever,10,8,0,0,0,0,10,10,0,7,...,0,10,10,10,10,10,0,0,7,8
kannequi,8,0,0,0,0,0,0,0,0,0,...,0,0,5,0,0,0,0,0,5,7
xkourokox,0,10,0,0,0,0,0,0,0,10,...,0,8,9,0,8,7,0,7,8,0


## Shows User hasn't Watched. Others have

In [10]:
matrix_Other_User_Ratings = matrix_Raw.loc[similar_users.index, ~alreadyWatched]
matrix_Other_User_Ratings

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60512,60544,60553,60627,60629,60736,60749,60772,61130,61134
User,,,,,,,,,,,,,,,,,,,,,
_Mei_,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
jasmine_kb08,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
xiuzuki,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
MysticMira,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
gellai,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Chaos_Griever,0,0,0,0,0,10,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
kannequi,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
xkourokox,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Aohau,10,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Recommendations' Avg Score by Similar Users

In [11]:
reccomended_Shows = matrix_Other_User_Ratings.mean(axis=0).sort_values(ascending=False)[:10]
df_Reccomendations = df_AnimeList.loc[reccomended_Shows.index].copy()
df_Reccomendations['Avg_Score'] = reccomended_Shows.values
df_Reccomendations

,Title,Avg_Score
MAL_id,,
11111,Another,7.5
4181,Clannad: After Story,6.6
7054,Kaichou wa Maid-sama!,6.2
28999,Charlotte,5.6
16894,Kuroko no Basket 2nd Season,5.5
28891,Haikyuu!! Second Season,5.2
9919,Ao no Exorcist,5.2
30276,One Punch Man,5.2
24833,Ansatsu Kyoushitsu,4.9


In [12]:
def GetPersonalTop(USER, n = 10):
    USER_top = matrix_Raw.loc[USER].sort_values(ascending=False)[:n]
    
    if USER_top.index.isin(df_AnimeList.index).all():
        df = df_AnimeList.loc[USER_top.index].copy()
    else:
        df = df_Raw[['MAL_id', 'Title']].copy().set_index('MAL_id').loc[USER_top.index].drop_duplicates()
    
    df['Personal_Score'] = USER_top.values
    df = df.rename(columns={'Title': f"{USER}'s Top {n}"})
    return df

    
def Recommend(USER, n = 10):
    similar_users = matrix_Users.loc[USER].sort_values(ascending=False)[1:n+1]
    alreadyWatched = matrix_Raw.loc[USER] > 0
    matrix_Other_User_Ratings = matrix_Raw.loc[similar_users.index, ~alreadyWatched]

    reccomended_Shows = matrix_Other_User_Ratings.mean(axis=0).sort_values(ascending=False)[:n]

    if reccomended_Shows.index.isin(df_AnimeList.index).all():
        df = df_AnimeList.loc[reccomended_Shows.index].copy()
    else:
        df = df_Raw[['MAL_id', 'Title']].copy().set_index('MAL_id').loc[reccomended_Shows.index].drop_duplicates()
        
    df['Avg_Recommenders_Score'] = reccomended_Shows.values
    df = df.rename(columns={'Title': f"Others' Top {n}"})
        
    return df

## Test Mainstream Viewer

In [13]:
USER = '-Norza-'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,-Norza-'s Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,31704,One Punch Man Road to Hero,10,37450,Seishun Buta Yarou wa Bunny Girl Senpai no Yum...,8.9
1,34626,Kono Subarashii Sekai ni Shukufuku wo 2 Kono ...,10,41389,Tonikaku Kawaii,8.5
2,30831,Kono Subarashii Sekai ni Shukufuku wo,10,4224,Toradora!,8.0
3,39547,Yahari Ore no Seishun Love Comedy wa Machigatt...,10,50265,Spy x Family,7.4
4,32937,Kono Subarashii Sekai ni Shukufuku wo 2,10,24833,Ansatsu Kyoushitsu,7.4
5,52578,Boku no Kokoro no Yabai Yatsu,10,40748,Jujutsu Kaisen,7.3
6,38000,Kimetsu no Yaiba,10,38329,Seishun Buta Yarou wa Yumemiru Shoujo no Yume ...,7.3
7,37999,Kaguya-sama wa Kokurasetai Tensai-tachi no Re...,10,15809,Hataraku Maou-sama!,7.0
8,23847,Yahari Ore no Seishun Love Comedy wa Machigatt...,10,39597,Sword Art Online: Alicization - War of Underworld,6.9
9,578,Hotaru no Haka,10,36474,Sword Art Online: Alicization,6.9


## Test Watched Only One Punch Man (Cold Start)

In [14]:
USER = 'EXUPUROOOSHUN'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,EXUPUROOOSHUN's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,30276,One Punch Man,10,28851,Koe no Katachi,3.8
1,1,Cowboy Bebop,0,22319,Tokyo Ghoul,2.8
2,35835,RErideD: Tokigoe no Derrida,0,27631,God Eater,2.0
3,35815,Boku dake ga Inai Machi Recaps,0,22199,Akame ga Kill!,1.9
4,35821,Chio-chan no Tsuugakuro,0,16498,Shingeki no Kyojin,1.9
5,35823,Demi-chan wa Kataritai: Demi-chan no Natsuyasumi,0,28121,Dungeon ni Deai wo Motomeru no wa Machigatteir...,1.9
6,35827,Otome*Domain The Animation,0,32281,Kimi no Na wa.,1.8
7,35828,Miira no Kaikata,0,31240,Re:Zero kara Hajimeru Isekai Seikatsu,1.8
8,35834,Angolmois: Genkou Kassenki,0,26243,Owari no Seraph,1.5
9,35838,Shoujo Shuumatsu Ryokou,0,1562,Yamato Nadeshiko Shichihenge♥,1.0


## Test Max Ranker : Everything 10

In [15]:
USER = 'Muhclis'
pd.concat([GetPersonalTop(USER,100).reset_index(), Recommend(USER,100).reset_index()], axis=1)

,MAL_id,Muhclis's Top 100,Personal_Score,MAL_id,Others' Top 100,Avg_Recommenders_Score
0,27363,Yakata Kannou Kitan,10,31240,Re:Zero kara Hajimeru Isekai Seikatsu,5.92
1,40746,Overflow,10,39587,Re:Zero kara Hajimeru Isekai Seikatsu 2nd Season,5.11
2,9487,Hitou Meguri Kakure Yu,10,33486,Boku no Hero Academia 2nd Season,5.09
3,32899,Watashi ga Motete Dousunda,10,4224,Toradora!,4.97
4,32895,Chijoku no Seifuku,10,31964,Boku no Hero Academia,4.95
...,...,...,...,...,...,...
95,36109,Boku dake no Hentai Kanojo Motto♥ The Animation,10,31765,Sword Art Online Movie: Ordinal Scale,2.45
96,1634,Ai Shimai Futari no Kajitsu,10,31181,Owarimonogatari,2.44
97,36121,Ikkyuu Nyuukon,10,29786,Shimoneta to Iu Gainen ga Sonzai Shinai Taikut...,2.42
98,15039,Ano Hi Mita Hana no Namae wo Bokutachi wa Mada...,10,40496,Maou Gakuin no Futekigousha: Shijou Saikyou no...,2.40


## AOT Fan

In [16]:
USER = 'Math082r'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,Math082r's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,28851,Koe no Katachi,10,52991,Sousou no Frieren,6.6
1,2251,Baccano!,10,2904,Code Geass: Hangyaku no Lelouch R2,6.4
2,40028,Shingeki no Kyojin: The Final Season,9,1575,Code Geass: Hangyaku no Lelouch,6.3
3,48583,Shingeki no Kyojin: The Final Season Part 2,9,47778,Kimetsu no Yaiba: Yuukaku-hen,6.3
4,3297,Aria the Origination,9,38000,Kimetsu no Yaiba,6.1
5,59571,Shingeki no Kyojin Movie: Kanketsu-hen - The L...,9,11843,Danshi Koukousei no Nichijou,6.0
6,51535,Shingeki no Kyojin: The Final Season - Kankets...,9,38691,Dr. Stone,5.9
7,4181,Clannad: After Story,9,49926,Kimetsu no Yaiba: Mugen Ressha-hen,5.9
8,39198,Kanata no Astra,8,40852,Dr. Stone: Stone Wars,5.7
9,8726,Soredemo Machi wa Mawatteiru,8,11111,Another,5.7


## Drama/Seinnen/Character Growth (Monogatari / Garden of Sinners / Vinland S2)

In [17]:
USER = 'Shizawari'
pd.concat([GetPersonalTop(USER, 15).reset_index(), Recommend(USER, 15).reset_index()], axis=1)

,MAL_id,Shizawari's Top 15,Personal_Score,MAL_id,Others' Top 15,Avg_Recommenders_Score
0,49387,Vinland Saga Season 2,10,42897,Horimiya,8.400000
1,57334,Dandadan,10,32281,Kimi no Na wa.,8.333333
2,53998,Bleach Sennen Kessen-hen - Ketsubetsu-tan,10,1535,Death Note,8.000000
3,33050,Fate stay night Movie Heavens Feel - III Spri...,10,45576,Mushoku Tensei: Isekai Ittara Honki Dasu Part 2,7.933333
4,37105,Grand Blue,10,23273,Shigatsu wa Kimi no Uso,7.800000
5,5205,Kara no Kyoukai Movie 7 Satsujin Kousatsu Go,10,38691,Dr. Stone,7.266667
6,9260,Kizumonogatari I Tekketsu-hen,10,28999,Charlotte,7.133333
7,48653,Yoru ni Kakeru,10,28171,Shokugeki no Souma,7.066667
8,56784,Bleach Sennen Kessen-hen - Soukoku-tan,10,5114,Fullmetal Alchemist: Brotherhood,7.000000
9,37521,Vinland Saga,10,44511,Chainsaw Man,6.866667


## Gooner

In [18]:
USER = 'omanko1'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,omanko1's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,49295,Chizuru-chan Kaihatsu Nikki,10,11827,Kuroinu: Kedakaki Seijo wa Hakudaku ni Somaru,7.3
1,50192,Akane wa Tsumare Somerareru,10,38470,Saimin Seishidou,6.7
2,59119,Imaria,10,24641,Baku Ane: Otouto Shibocchau zo! The Animation,6.3
3,33291,Pinkerton,9,36720,Youkoso! Sukebe Elf no Mori e,6.3
4,32587,Shoujo Ramune,9,39042,Gaki ni Modotte Yarinaoshi!!!,6.0
5,37287,Tsuma Netori: Ryoujoku Rinne,9,48698,Mako-chan Kaihatsu Nikki,5.6
6,35581,Oide yo! Mizuryuu Kei Land,9,33322,JK Bitch ni Shiboraretai,5.5
7,23479,Itadaki! Seieki♥ Special,9,38866,Seikatsu Shuukan The Animation,5.4
8,2539,Itazura The Animation,9,29575,Mankitsu Happening,5.4
9,42224,Shikiyoku Infinite,9,21829,Fella Pure: Mitarashi-san Chi no Jijou The Ani...,5.4


# Review
Compared to content based, this method feels more like a black box approach.

It relies on the shrared ratings between users that will often be difficult to visually interpret considering the matrix's size. More than that, most of what is recommended is Mainstream due to majority ruling.

The only most notable distinction we could make was that a Porn viewer is correctly recommendeded explicit content from other users. 

# Item-Item Approach

In [19]:
matrix_Anime = pd.DataFrame(cosine_similarity(matrix_Raw.T), index=matrix_Raw.columns, columns=matrix_Raw.columns)
matrix_Anime.head()

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60512,60544,60553,60627,60629,60736,60749,60772,61130,61134
MAL_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.567642,0.339106,0.209620,0.163519,0.226838,0.217057,0.179232,0.221987,0.395254,...,0.0,0.058720,0.0,0.084598,0.0,0.084598,0.0,0.075198,0.084598,0.084598
5,0.567642,1.000000,0.281238,0.148063,0.078890,0.124740,0.133346,0.148867,0.116468,0.303199,...,0.0,0.124752,0.0,0.127808,0.0,0.143784,0.0,0.159760,0.127808,0.127808
6,0.339106,0.281238,1.000000,0.274011,0.206740,0.246597,0.135410,0.194930,0.248160,0.241715,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
7,0.209620,0.148063,0.274011,1.000000,0.140211,0.139025,0.220320,0.357124,0.250726,0.113382,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
8,0.163519,0.078890,0.206740,0.140211,1.000000,0.278984,0.172554,0.275682,0.165322,0.068461,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000


## Test Harem : Rent a Girlfirend

Expecting Subsequent Seasons + Other harem Anime 

In [20]:
def ReccommendByItem(ANIME_ID : int, n = 10):
    similar_ID = matrix_Anime.loc[ANIME_ID].sort_values(ascending=False)[:n+1]
    return pd.concat([df_Raw.set_index('MAL_id').loc[similar_ID.index]['Title'].drop_duplicates(), similar_ID], axis=1)

In [21]:
ReccommendByItem(40839)

,Title,40839
MAL_id,,
40839,Kanojo Okarishimasu,1.000000
37450,Seishun Buta Yarou wa Bunny Girl Senpai no Yum...,0.596787
38101,5-toubun no Hanayome,0.585130
39547,Yahari Ore no Seishun Love Comedy wa Machigatt...,0.579655
39783,5-toubun no Hanayome ∬,0.572874
40591,Kaguya-sama wa Kokurasetai Tensai-tachi no Ren...,0.556714
40938,Hige wo Soru Soshite Joshikousei wo Hirou,0.547862
39535,Mushoku Tensei Isekai Ittara Honki Dasu,0.545106
41389,Tonikaku Kawaii,0.542732


## Test Niche Gambling : Kakegurui
Expecting shows related to competition.

In [22]:
ReccommendByItem(34933)

,Title,34933
MAL_id,,
34933,Kakegurui,1.000000
37086,Kakegurui××,0.765335
37999,Kaguya-sama wa Kokurasetai Tensai-tachi no Re...,0.583592
40591,Kaguya-sama wa Kokurasetai Tensai-tachi no Ren...,0.557645
37450,Seishun Buta Yarou wa Bunny Girl Senpai no Yum...,0.555274
28851,Koe no Katachi,0.542097
38000,Kimetsu no Yaiba,0.535511
35849,Darling in the FranXX,0.528232
32282,Shokugeki no Souma Ni no Sara,0.523302
